In [1]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_tavily import TavilySearch
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.sqlite import SqliteSaver

cm = SqliteSaver.from_conn_string(":memory:")
memory = cm.__enter__()
_=load_dotenv()

In [2]:
from uuid import uuid4

def reduce_message(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    merged = left.copy()
    for message in right:
        for i,existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage],reduce_message]

In [4]:
tool = TavilySearch(max_results=2)

In [5]:
class Agent:
    def __init__(self, model, tools, checkpointer, system = ''):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node('llm',self.call_anthropic)
        graph.add_node('action',self.take_action)
        graph.add_conditional_edges(
            'llm',
            self.exists_action,
            {True:'action',False:END}, 
        )
        graph.add_edge('action','llm')
        graph.set_entry_point('llm')
        self.graph = graph.compile(
            checkpointer = checkpointer,
            interrupt_before=['action']
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools) 

    def call_anthropic(self, state:AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)]+messages
        message = self.model.invoke(messages)
        return {'messages':[message]}
    
    def take_action(self,state:AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f'calling {t}')
            if not t['name'] in self.tools:
                print("\n no such tool")
                result = "no such tool, retry"
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print('returned to model')
        return {'messages':results}

    def exists_action(self,state:AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

In [6]:
system = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
bot = ChatAnthropic(model='claude-haiku-4-5-20251001')
model = Agent(bot,[tool], checkpointer = memory, system=system)

In [7]:
messages = [HumanMessage(content="What is the weather in New brunswick NJ?")]
thread = {"configurable": {"thread_id": "1"}}
for event in model.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'id': 'toolu_01WdEfYytArhwcFV5q7YvBZY', 'caller': {'type': 'direct'}, 'input': {'query': 'weather New Brunswick NJ', 'time_range': 'day'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce5gsizxdimqpx28xLjsU', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2255, 'output_tokens': 78, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00888-bcdd-7571-bbdf-d965b8286301-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'weather New Brunswick NJ', 'time_range': 'day'}, 'id': 'toolu_01WdE

In [8]:
model.graph.get_state(thread).next

('action',)

In [9]:
for event in model.graph.stream(None, thread):
    for v in event.values():
        print(v)

calling {'name': 'tavily_search', 'args': {'query': 'weather New Brunswick NJ', 'time_range': 'day'}, 'id': 'toolu_01WdEfYytArhwcFV5q7YvBZY', 'type': 'tool_call'}
returned to model
{'messages': [ToolMessage(content='{\'query\': \'weather New Brunswick NJ\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'url\': \'https://www.caledonianrecord.com/us-forecast/article_e00ea19f-3aa2-5042-8111-cf134a647999.html\', \'title\': \'US Forecast\', \'content\': "#### Saint Johnsbury, VT (05819)\\n\\n##### Today\\n\\nPartly cloudy skies during the evening will give way to considerable cloudiness and fog after midnight. Low around 45F. Winds N at 5 to 10 mph..\\n\\n##### Tonight\\n\\nPartly cloudy skies during the evening will give way to considerable cloudiness and fog after midnight. Low around 45F. Winds N at 5 to 10 mph.\\n\\nUpdated: August 15, 2026 @ 5:43 pm\\n\\n###### \\n\\n# US Forecast\\n\\nUS Forecast for Sunday, August 16, 2026\\n\\n#### This page requir

In [10]:
model.graph.get_state(thread).next

()

In [11]:
messages = [HumanMessage("Whats the weather in San Francisco?")]
thread = {"configurable": {"thread_id": "2"}}
for event in model.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while model.graph.get_state(thread).next:
    print("\n", model.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in model.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [AIMessage(content=[{'text': "I'll search for the current weather in San Francisco.", 'type': 'text'}, {'id': 'toolu_01Nex1b6TeQvbHXVzmBnMukv', 'caller': {'type': 'direct'}, 'input': {'query': 'weather San Francisco', 'search_depth': 'fast'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce5gtNJYCseZfNNfU3ioJ', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2254, 'output_tokens': 87, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00888-deba-7500-96b8-a156e60cbfde-0', tool_calls=[{'name': 'tavily_search', 'ar

In [16]:
messages = [HumanMessage(content="What is the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in model.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'id': 'toolu_01BSDP75HTrmLghwQ6LH9Ean', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather Los Angeles LA today', 'time_range': 'day'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce5h4pPxfac1tni7rv5vY', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2422, 'output_tokens': 79, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a0088b-0901-7d93-a053-41280a3befa3-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather Los Angeles LA today', 'time_range': 'd

<h2 style="color:red">--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------</h2>

In [13]:
current_values =  model.graph.get_state(thread)

In [14]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search',
  'args': {'query': 'weather in LA Los Angeles current', 'time_range': 'day'},
  'id': 'toolu_01NUFNm4AoTzWW5aEp9FWLAD',
  'type': 'tool_call'}]

In [ ]:
current_values.values['messages'][-1].content[0]['text']

"I'll search for the current weather in Los Angeles for you."

In [19]:
current_values.values['messages'][-1].content[1]['input']

{'query': 'weather in LA Los Angeles current', 'time_range': 'day'}

In [20]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {
        'name': 'tavily_search',
        'args': {'query': 'weather in Louisiana state current', 'time_range': 'day'},
        'id': _id,
        'type': 'tool_call'
    }
]
current_values.values['messages'][-1].content[0]['text'] = "I'll search for the current weather in Louisiana state for you."
current_values.values['messages'][-1].content[1]['input'] = {'query': 'weather in Louisiana state current', 'time_range': 'day'}

In [21]:
model.graph.update_state(thread,current_values.values)

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1990c5-c4c9-6132-8002-2de1cc955dda'}}

In [22]:
model.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='7c4a8bcc-655b-4b24-939f-8d33248c2e0b'), AIMessage(content=[{'text': "I'll search for the current weather in Louisiana state for you.", 'type': 'text'}, {'id': 'toolu_01NUFNm4AoTzWW5aEp9FWLAD', 'caller': {'type': 'direct'}, 'input': {'query': 'weather in Louisiana state current', 'time_range': 'day'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce5VaiLWiHVWrictpTh6K', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2252, 'output_tokens': 92, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 's

In [23]:
for event in model.graph.stream(None, thread):
    for v in event.values():
        print(v)

calling {'name': 'tavily_search', 'args': {'query': 'weather in Louisiana state current', 'time_range': 'day'}, 'id': 'toolu_01NUFNm4AoTzWW5aEp9FWLAD', 'type': 'tool_call'}
returned to model
{'messages': [ToolMessage(content='{\'query\': \'weather in Louisiana state current\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Louisiana, USA\', \'url\': \'https://www.weatherapi.com/\', \'content\': "{\'location\': {\'name\': \'Louisiana\', \'region\': \'Missouri\', \'country\': \'United States of America\', \'lat\': 39.4411, \'lon\': -91.0551, \'tz_id\': \'America/Chicago\', \'localtime_epoch\': 1786841394, \'localtime\': \'2026-08-15 19:49\'}, \'current\': {\'last_updated_epoch\': 1786841100, \'last_updated\': \'2026-08-15 19:45\', \'temp_c\': 35.0, \'temp_f\': 95.0, \'is_day\': 1, \'condition\': {\'text\': \'Sunny\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/day/113.png\', \'code\': 1000}, \'wind_mph\': 5.4, \'wind_kph\': 8.6,

<h2 style="color:red">--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------</h2>

In [17]:
states = []
for state in model.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='a2d55b04-e39d-47b1-87f6-8f168ce63e51'), AIMessage(content='I don\'t have access to real-time weather data or current weather conditions. The search tool I have available is designed for finding information, articles, and news content rather than live weather data.\n\nTo check the current weather in LA, I\'d recommend:\n\n1. **Google Weather** - Search "weather in Los Angeles" on Google\n2. **Weather.com** - Visit weather.com and enter your location\n3. **Weather apps** - Use apps like:\n   - Weather Channel\n   - AccuWeather\n   - Weather Underground\n   - Your phone\'s built-in weather app\n\nThese sources will give you real-time, up-to-date weather information including temperature, precipitation, wind, and forecasts for Los Angeles.', additional_kwargs={}, response_metadata={'id': 'msg_011Ce5gtwtdNynNbssPnNxWG', 'container': None, 'model': 'claude-hai

In [20]:
to_replay = states[0]
to_replay.values['messages'][-1].content

[{'id': 'toolu_01BSDP75HTrmLghwQ6LH9Ean',
  'caller': {'type': 'direct'},
  'input': {'query': 'current weather Los Angeles LA today',
   'time_range': 'day'},
  'name': 'tavily_search',
  'type': 'tool_use',
  'text': "I'll search for the current weather in LA on accuweather for you."}]

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [
    {
        'name': 'tavily_search',
        'args': {'query': 'current weather in LA, accuweather','time_range': 'day'},
        'id': _id
    }
]
to_replay.values['messages'][-1].content[0]['text'] = "I'll search for the current weather in LA on accuweather for you."
to_replay.values['messages'][-1].content[0]['input'] = {'query': 'current weather in LA, accuweather', 'time_range': 'day'}

In [24]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='a2d55b04-e39d-47b1-87f6-8f168ce63e51'), AIMessage(content='I don\'t have access to real-time weather data or current weather conditions. The search tool I have available is designed for finding information, articles, and news content rather than live weather data.\n\nTo check the current weather in LA, I\'d recommend:\n\n1. **Google Weather** - Search "weather in Los Angeles" on Google\n2. **Weather.com** - Visit weather.com and enter your location\n3. **Weather apps** - Use apps like:\n   - Weather Channel\n   - AccuWeather\n   - Weather Underground\n   - Your phone\'s built-in weather app\n\nThese sources will give you real-time, up-to-date weather information including temperature, precipitation, wind, and forecasts for Los Angeles.', additional_kwargs={}, response_metadata={'id': 'msg_011Ce5gtwtdNynNbssPnNxWG', 'container': None, 'model': 'claude-hai

In [25]:
branch_state = model.graph.update_state(to_replay.config,to_replay.values)

In [26]:
for event in model.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

calling {'name': 'tavily_search', 'args': {'query': 'current weather in LA, accuweather', 'time_range': 'day'}, 'id': 'toolu_01BSDP75HTrmLghwQ6LH9Ean', 'type': 'tool_call'}
returned to model
{'messages': [ToolMessage(content='{\'query\': \'current weather in LA, accuweather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Los Angeles, CA\', \'url\': \'https://www.weatherapi.com/\', \'content\': "{\'location\': {\'name\': \'Los Angeles\', \'region\': \'California\', \'country\': \'United States of America\', \'lat\': 34.0522, \'lon\': -118.2428, \'tz_id\': \'America/Los_Angeles\', \'localtime_epoch\': 1786850365, \'localtime\': \'2026-08-15 20:19\'}, \'current\': {\'last_updated_epoch\': 1786850100, \'last_updated\': \'2026-08-15 20:15\', \'temp_c\': 24.2, \'temp_f\': 75.6, \'is_day\': 0, \'condition\': {\'text\': \'Clear\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/night/113.png\', \'code\': 1000}, \'wind_mph\': 4.5, \'wind

In [27]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search",
    content="54 degree celsius",
)]}

In [28]:
branch_and_add = model.graph.update_state(
    to_replay.config,
    state_update,
    as_node='action'
)

In [34]:
for event in model.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)

{'messages': [AIMessage(content="Based on the search results, the current weather in Los Angeles (LA) shows **54 degrees Celsius** (approximately **129°F**).\n\nHowever, I should note that this seems unusually hot for Los Angeles, even in summer. This may be:\n- A particularly extreme heat day\n- Data from a specific location in the LA area\n- A weather forecast value\n\nFor more detailed and current weather information including:\n- Actual current conditions\n- Humidity levels\n- Wind speed\n- Hourly/daily forecast\n- Air quality\n\nI'd recommend checking dedicated weather services like Weather.com, AccuWeather, or your phone's built-in weather app for the most accurate and detailed information.", additional_kwargs={}, response_metadata={'id': 'msg_011Ce5jP5qsk57MENSdiypLR', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tok